In [1]:
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch("http://localhost:9200")
print(es.info())

{'name': '053ec4d8388f', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'e7EmbiJYRsGOwKDitJv50g', 'version': {'number': '8.15.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179', 'build_date': '2024-08-05T10:05:34.233336849Z', 'build_snapshot': False, 'lucene_version': '9.11.1', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [2]:
movies = [
    {"title": "Interstellar", "genre": "SciFi", "year": 2014, "plot": "Astronauts travel through a wormhole to save humanity.", "rating": 8.7},
    {"title": "Pulp Fiction", "genre": "Crime", "year": 1994, "plot": "Intertwined stories of crime in Los Angeles.", "rating": 8.9},
    {"title": "Up", "genre": "Animation", "year": 2009, "plot": "An old man flies his house away with balloons.", "rating": 8.3},
]

def gen(docs, index):
    for i, d in enumerate(docs):
        yield {"_index": index, "_id": i, "_source": d}

ok, errors = helpers.bulk(es, gen(movies, "movies_bulk"))
print("Indexed:", ok, "Errors:", errors)

Indexed: 3 Errors: []


In [1]:
from elasticsearch import Elasticsearch
es = Elasticsearch("http://localhost:9200")
INDEX = "imdb_reviews"

In [2]:
r = es.search(index=INDEX, query={
    "bool": {
        "must":   [{"match": {"text": "best cinematography"}}],
        "filter": [{"term":  {"sentiment": "positive"}}]
    }
}, size=5)

for h in r["hits"]["hits"]:
    print(round(h["_score"], 2), h["_source"]["sentiment"], "|", h["_source"]["text"][:150])

8.5 positive | This is one of the best movies I've ever seen. It has very good acting by Hanks, Newman, and everyone else. Definitely Jude Law's best performance. Th
7.48 positive | Surely one of the best British films ever made, if not one of the best films ever made anywhere. Script, cinematography, direction and acting in a cla
7.45 positive | I went to see Hamlet because I was in between jobs. I figured 4 hours would be great, I've been a fan of Branagh; Dead Again, Henry V. I was completel
7.01 positive | this is best comedy i ever seen! but not all can understand this you must be from Georgia to understand this amazing movie! :) overall one of best fil
7.01 positive | Absolutely enjoyable singing and dancing movie starring Frank Sinatra and gen Kelly, as well as Kathryn Grayson.<br /><br />The film won and Oscar for


In [3]:
r = es.search(index=INDEX, query={
    "bool": {
        "must":   [{"match": {"text": {"query": "best cinematography", "operator": "and"}}}],
        "filter": [{"term":  {"sentiment": "positive"}}]
    }
}, size=5)

for h in r["hits"]["hits"]:
    print(round(h["_score"], 2), "|", h["_source"]["text"][:150])

8.5 | This is one of the best movies I've ever seen. It has very good acting by Hanks, Newman, and everyone else. Definitely Jude Law's best performance. Th
7.48 | Surely one of the best British films ever made, if not one of the best films ever made anywhere. Script, cinematography, direction and acting in a cla
7.45 | I went to see Hamlet because I was in between jobs. I figured 4 hours would be great, I've been a fan of Branagh; Dead Again, Henry V. I was completel
7.01 | this is best comedy i ever seen! but not all can understand this you must be from Georgia to understand this amazing movie! :) overall one of best fil
7.01 | Absolutely enjoyable singing and dancing movie starring Frank Sinatra and gen Kelly, as well as Kathryn Grayson.<br /><br />The film won and Oscar for


In [4]:
for h in r["hits"]["hits"]:
    t = h["_source"]["text"].lower()
    print("cinematograph" in t, len(t), "chars")

True 516 chars
True 376 chars
True 1162 chars
True 812 chars
True 730 chars


In [5]:
r = es.search(index=INDEX, size=0, aggs={
    "by_sentiment": {
        "terms": {"field": "sentiment"},
        "aggs": {
            "length_stats": {"stats": {"field": "length"}},
            "length_buckets": {"histogram": {"field": "length", "interval": 1000}}
        }
    }
})

for b in r["aggregations"]["by_sentiment"]["buckets"]:
    s = b["length_stats"]
    print(f'{b["key"]}: {b["doc_count"]} reviews, avg {s["avg"]:.0f} chars, min {s["min"]:.0f}, max {s["max"]:.0f}')
    for h in b["length_buckets"]["buckets"][:6]:
        print(f'   {h["key"]:>5.0f}-{h["key"]+999:.0f} chars: {h["doc_count"]}')

negative: 12500 reviews, avg 1303 chars, min 52, max 8969
       0-999 chars: 6454
    1000-1999 chars: 4003
    2000-2999 chars: 1228
    3000-3999 chars: 458
    4000-4999 chars: 219
    5000-5999 chars: 121
positive: 12500 reviews, avg 1347 chars, min 70, max 13704
       0-999 chars: 6397
    1000-1999 chars: 3822
    2000-2999 chars: 1279
    3000-3999 chars: 568
    4000-4999 chars: 259
    5000-5999 chars: 149


In [6]:
r = es.search(index=INDEX, query={
    "bool": {
        "must":   [{"multi_match": {"query": "brilliant acting", "fields": ["text"]}}],
        "should": [{"match": {"text": {"query": "brilliant", "boost": 2}}}]
    }
}, size=5)

for h in r["hits"]["hits"]:
    print(round(h["_score"], 2), "|", h["_source"]["text"][:150])

18.37 | What surprised me most about this film was the sheer audience it attracted. Similar films such as Anita and Me have never caused as much hype as this 
17.74 | From the start I knew I would be in for the best movie watching experience of my life. The idea of two giant robots, manned by brilliant humans, fight
17.64 | This is a great Canadian comedy series. The movie tells of how the stars Jean Paul Tremblay-(Julian) and the head writer of the show and his buddies R
17.12 | Govind Nihalani's directorial venture of Vijay Tendulkar's novel is brilliant. Om Puri plays an inspector Velankar who is forced to protect underworld
17.12 | Brilliant film, the next best film to The Drunken Master (Jackie Chan). I recently bought it on an original VHS and i haven't seen this film for 15 ye


In [7]:
r = es.search(index=INDEX, query={
    "match_phrase": {"text": "waste of time"}
}, size=5)

print("Total matches:", r["hits"]["total"]["value"])
for h in r["hits"]["hits"]:
    print(round(h["_score"], 2), h["_source"]["sentiment"], "|", h["_source"]["text"][:150])

Total matches: 787
6.28 negative | Do not waste your time watching this! Unless you want to study it for all the wrong things you shouldn't do to make a good film. I am not one to usual
6.17 negative | Horrible waste of time - bad acting, plot, directing. This is the most boring movie EVER! There are bad movies that are fun (Freddy vs. Jason), and th
6.05 negative | <br /><br />What an absolutely crappy film this is. How or why this movie was made and what the hell Billy Bob Thornton and Charlize Theron were doing
5.97 negative | What's up with Robert "Pretentious" Altman? Was he saving on lighting? Everything was so dark in this boring movie that it was laughable. I mean, have
5.89 negative | Watching this movie was a waste of time. I was tempted to leave in the middle of the movie, but I resisted. I don't know what Ridley Scott intended, b


In [8]:
r = es.search(index=INDEX, size=0, query={"match_phrase": {"text": "waste of time"}},
              aggs={"s": {"terms": {"field": "sentiment"}}})
print(r["aggregations"]["s"]["buckets"])

[{'key': 'negative', 'doc_count': 739}, {'key': 'positive', 'doc_count': 48}]


In [9]:
print("SOURCE REVIEW:", es.get(index=INDEX, id=0)["_source"]["text"][:200], "\n")

r = es.search(index=INDEX, query={
    "more_like_this": {
        "fields": ["text"],
        "like": [{"_index": INDEX, "_id": "0"}],
        "min_term_freq": 1,
        "max_query_terms": 25
    }
}, size=5)

for h in r["hits"]["hits"]:
    print(round(h["_score"], 2), "|", h["_source"]["text"][:150])

SOURCE REVIEW: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev 

35.06 | I am Curious (Yellow) (a film, in near Seussical rhyme, is said right at the start to be available in two versions, Yellow and Blue) was one of those 
28.83 | I don't know how or why this film has a meager rating on IMDb. This film, accompanied by "I am Curious: Blue" is a masterwork.<br /><br />The only thi
28.36 | I didn't expect to like this film as much as I did. I got it simply because I saw it on the list of Top 25 Most Controversial Films of All Time. It di
27.93 | Oh, brother...after hearing about this ridiculous film for umpteen years all I can think of is that old Peggy Lee song..<br /><br />"Is that all there
25.61 | Fascinating I approached I Am Curious (Yellow) and it's companion piece with great trepidation. I'd read numerous reports on its widely to